In [2]:
import random

import numpy as np
from torch.utils.data import Dataset
from seqeval.metrics import f1_score, precision_score, recall_score
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
    TrainingArguments,
    Trainer,
)

In [4]:
import sys
sys.path.append("../src")  # adjust relative path based on where your notebook lives

from data_utils import load_funsd_doc, ANN_DIR

In [5]:
LABEL_LIST = ["question", "answer", "header", "other"]
label2id = {label: i for i, label in enumerate(LABEL_LIST)}
id2label = {i: label for label, i in label2id.items()}

In [6]:
# apply_ocr=False because we're supplying our own words/boxes (FUNSD ground truth),
# not asking the processor to run Tesseract internally
processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base", apply_ocr=False
)

In [7]:
def get_train_val_split(val_ratio=0.15, seed=42):
    """
    Split the 149 FUNSD training docs into train/val subsets.
    FUNSD's official 'testing_data' folder is kept untouched -
    reserved for final reporting only, not used here.
    """
    stems = [f.stem for f in ANN_DIR.glob("*.json")]

    rng = random.Random(seed)  # fixed seed -> reproducible split across runs
    rng.shuffle(stems)

    val_size = int(len(stems) * val_ratio)
    val_stems = stems[:val_size]
    train_stems = stems[val_size:]

    return train_stems, val_stems

In [8]:
class FUNSDLayoutDataset(Dataset):
    """
    Wraps FUNSD documents for LayoutLMv3 token classification.
    Each __getitem__ call loads one document, normalizes its boxes,
    and runs it through the processor to produce model-ready tensors.
    """

    def __init__(self, stems):
        self.stems = stems

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]
        image, words, boxes, labels = load_funsd_doc(stem)
        label_ids = [label2id[l] for l in labels]

        encoding = processor(
            image,
            words,
            boxes=boxes,
            word_labels=label_ids,
            truncation=True,
            padding="max_length",   # pad to model max length here; simplifies
            max_length=512,          # collator usage for this first pass
            return_tensors="pt",
        )

        # processor adds a batch dim (shape [1, ...]) - squeeze it off, since
        # Dataset __getitem__ must return a single unbatched example
        return {k: v.squeeze(0) for k, v in encoding.items()}

In [16]:
from sklearn.metrics import classification_report, f1_score

def compute_metrics(eval_pred):
    """
    Compute macro and micro F1 scores for token classification.
    It ignores tokens with label -100 (the default ignore index for token classification).
    Takes eval_pred, a tuple of (logits, labels), where logits are the model outputs and labels are the true labels.
    Returns a dictionary with 'f1_macro' and 'f1_micro' keys.
    """

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []
    for pred_row, label_row in zip(predictions, labels):
        for p, l in zip(pred_row, label_row):
            if l != -100:
                true_predictions.append(id2label[p])
                true_labels.append(id2label[l])

    return {
        "f1_macro": f1_score(true_labels, true_predictions, average="macro"),
        "f1_micro": f1_score(true_labels, true_predictions, average="micro"),
    }

In [17]:
train_stems, val_stems = get_train_val_split()
print(f"Train docs: {len(train_stems)}, Val docs: {len(val_stems)}")

Train docs: 127, Val docs: 22


In [18]:
train_dataset = FUNSDLayoutDataset(train_stems)
val_dataset = FUNSDLayoutDataset(val_stems)

In [19]:
model = LayoutLMv3ForTokenClassification.from_pretrained(
        "microsoft/layoutlmv3-base",
        num_labels=len(LABEL_LIST),
        id2label=id2label,
        label2id=label2id,
    )

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 5738.87it/s]
[transformers] LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
training_args = TrainingArguments(
        output_dir="./checkpoints/layoutlmv3-funsd",
        per_device_train_batch_size=2,   # small - 6GB VRAM constraint
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,   # effective batch size = 2*4 = 8
        num_train_epochs=10,
        learning_rate=5e-5,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        fp16=True,                        # mixed precision - fits 6GB VRAM
        logging_steps=10,
        report_to="none",                 # disable wandb/etc auto-logging
    )


In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    )


In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro
1,3.430235,0.808316,0.598492,0.701863
2,2.284074,0.718306,0.645721,0.747205
3,1.854384,0.644934,0.693274,0.782298
4,1.204223,0.663347,0.701473,0.784783
5,0.848696,0.721802,0.701159,0.784472
6,0.726433,0.748136,0.713755,0.787888
7,0.618848,0.717021,0.721674,0.802795
8,0.463028,0.770783,0.721640,0.798447
9,0.349734,0.805958,0.727193,0.799689
10,0.285813,0.814579,0.725445,0.796894


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.19s/it]


TrainOutput(global_step=160, training_loss=1.2009723603725433, metrics={'train_runtime': 463.7595, 'train_samples_per_second': 2.738, 'train_steps_per_second': 0.345, 'total_flos': 334769785282560.0, 'train_loss': 1.2009723603725433, 'epoch': 10.0})

In [27]:
metrics = trainer.evaluate()
print(metrics)

Training Loss,Validation Loss,Epoch,F1 Macro,F1 Micro
0.285813,0.805958,10,0.727193,0.799689


{'eval_loss': 0.8059581518173218, 'eval_f1_macro': 0.72719316890668, 'eval_f1_micro': 0.7996894409937888}


In [28]:
from sklearn.metrics import classification_report

predictions, labels, _ = trainer.predict(val_dataset)
preds = np.argmax(predictions, axis=-1)

true_predictions = []
true_labels = []
for pred_row, label_row in zip(preds, labels):
    for p, l in zip(pred_row, label_row):
        if l != -100:
            true_predictions.append(id2label[p])
            true_labels.append(id2label[l])

print(classification_report(true_labels, true_predictions))

              precision    recall  f1-score   support

      answer       0.78      0.93      0.85      1336
      header       0.81      0.48      0.60       257
       other       0.72      0.49      0.58       597
    question       0.86      0.89      0.87      1030

    accuracy                           0.80      3220
   macro avg       0.79      0.70      0.73      3220
weighted avg       0.80      0.80      0.79      3220

